In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

### What does the CSV contain

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Read the CSV directly from Google Drive
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/startup_founder_burnout_2026.csv")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
print(df.head(5))

In [ ]:
print(f'Shape : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Duplicates : {df.duplicated().sum()}')

print(f'\nNumeric columns : {len(df.select_dtypes(include=["number"]).columns)}')
print(f'\nCategorical columns : {len(df.select_dtypes(exclude=["number"]).columns)}')

Analysis of Basic Commands:

We can see that the dataset is 50,000 rows by 29 columns. There are no missing values or duplicates meaning that we do not have to do any data cleaning. Looking deeper into the column specifics we can see that 21 of the 29 columns contain numeric values and the remaining 8 values are categorical columns.

**Feature Variables**

We want to create three main variables:

Shutdown Probability - A number between 0 and 1 that represents how close a startup is to shutting down. We will think of this variable as a risk meter.

Shutdown Risk - A number between 0 and 1 as well, but it will be bucketed into three different tiers (Low, Medium, High). This makes it more readable rather looking at the raw number.

Startup Failure Flag - This is a binary situation (yes or no). If the startup failed that is a 1 (yes), 0 means no. This is the most straightforward thing we're trying to do when trying to predict.

In [ ]:
print(f'Shutdown Probability mean={df["Shutdown_Probability"].mean():.4f}, std={df["Shutdown_Probability"].std():.4f}')

print(f'Shutdown Risk distribution: {df["Shutdown_Risk"].value_counts().to_dict()}')

print(f'Startup Failure Flag failure rate: {df["Startup_Failure_Flag"].mean()*100:.1f}%')

**What Makes a Founder?**

In [ ]:
fig, axes = plt.subplots(2,3, figsize=(17,9))
fig.suptitle('Startup Founder Profiles - 50,000 Founders', fontsize = 14, fontweight = 'bold')

# Founder type distribution
ft = df['Founder_Type'].value_counts()
FT_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
axes[0,0].barh(ft.index,ft.values, color=FT_COLORS, edgecolor='white', height=0.5)
for i, (t,v) in enumerate(zip(ft.index,ft.values)):
    axes[0,0].text(v+100, i, f'{v:,}', va = 'center', fontsize = 6)
axes[0,0].set_title('Founder Types')
axes[0,0].set_xlabel('Number of Founders')

# Industry
ind = df['Industry'].value_counts()
axes[0,1].barh(ind.index,ind.values, color = '#2ca02c', edgecolor='white', height=0.5)
for i, (t,v) in enumerate(zip(ind.index,ind.values)):
    axes[0,1].text(v+20, i, f'{v:,}', va = 'center', fontsize = 6)
axes[0,1].set_title('Industries')
axes[0,1].set_xlabel('Founders')

# Funding stage
Stage_Order = ['Pre-Seed','Seed','Series A','Series B','Series C','Bootstrapped']
Stage_Colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
fs = df['Funding_Stage'].value_counts().reindex(Stage_Order)
axes[0,2].bar(fs.index, fs.values, color=Stage_Colors, edgecolor='white', width=0.5)
axes[0,2].tick_params(axis='x', rotation=25)
for i, (s,v) in enumerate(fs.items()):
    axes[0,2].text(i, v+100, f'{v:,}', ha='center', fontsize=6)
axes[0,2].set_title('Funding Stages')
axes[0,2].set_ylabel('Founders')

# Founder age
axes[1,0].hist(df['Founder_Age'], bins = 40, color = '#1f77b4', edgecolor='white', linewidth=0.3)
axes[1,0].axvline(df['Founder_Age'].mean(), color = '#ff7f0e', linestyle = '--', linewidth = 1.8,
                  label = f'Mean: {df["Founder_Age"].mean():.1f} yrs')
axes[1,0].set_title('Founder Age Distribution')
axes[1,0].set_xlabel('Age (years)')
axes[1,0].set_ylabel('Count')
axes[1,0].legend(fontsize = 9)

# Work hours distribution
axes[1,1].hist(df['Weekly_Work_Hours'], bins = 40, color = '#2ca02c', edgecolor='white', linewidth=0.3)
axes[1,1].axvline(df['Weekly_Work_Hours'].mean(), color = '#ff7f0e', linestyle = '--', linewidth = 1.8,
                  label = f'Mean: {df["Weekly_Work_Hours"].mean():.1f} hrs')
axes[1,1].set_title('Weekly Work Hours Distribution')
axes[1,1].set_xlabel('Hours (per week)')
axes[1,1].set_ylabel('Count')
axes[1,1].legend(fontsize = 8)

# Economic climate
CLIMATE_PAL = {'Bull Market' : '#2ca02c', 'Stable Economy' : '#d62728', 'Funding Winter' : '#9467bd', 'Recession' : '#8c564b'}
ec = df['Economic_Climate'].value_counts()
axes[1,2].pie(ec.values, labels=ec.index,
              colors=[CLIMATE_PAL.get(k, '#7f7f7f') for k in ec.index],
              autopct='%1.1f%%', startangle=90,
              wedgeprops={'edgecolor': 'white', 'linewidth': 0.5},
              textprops={'fontsize': 9})
axes[1,2].set_title('Economic Distribution')

plt.tight_layout()
plt.show()

Burnout Percentage

In [4]:
burnout_target = df['Founder_Burnout_Flag']
burnout_features = df.drop(['Founder_Burnout_Flag','Startup_Failure_Flag', 'Shutdown_Probability', 'Shutdown_Risk'], axis = 1)
burnout_features_binary = pd.getdummies(burnout_features, drop_first = True)

X_train_b, X_test_b, y_train_b, _y_test_b = train_test_split(burnout_features_binary, burnout_target, test_size = 0.2, random_state = 42)

scaler_b = StandardScaler()
X_train_b_scaled = scaler_b.fit_transform(X_train_b)
X_test_b_scaled = scaler_b.transform(X_test_b)

burnout_prob = xgb_burnout.predict_proba(scaler_b.transform(burnout_features_binary))[:,1]
df['Burnout_Percentage'] = burnout_proba * 100

NameError: name 'df' is not defined

Pytorch

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# Creating feature and target

feature = df.drop(['Startup_Failure_Flag', 'Shutdown_Probability', 'Shutdown_Risk'], axis=1)
target = df['Startup_Failure_Flag']

In [ ]:
#Train Test Split and Scaling
featues_binary = pd.get_dummies(feature, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(featues_binary, target, test_size=0.2, random_state=42)

scaler = StandardScaler()

x_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(X_test)

##Binary Classification


#Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

lrmodel = LogisticRegression(random_state=42)
lrmodel.fit(x_scaled, y_train)
y_pred = lrmodel.predict(x_test_scaled)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred, labels=lrmodel.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=lrmodel.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix (Logistic Regression)')
plt.show()

# **XGBoost**

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score

In [ ]:
#XG Boost Model

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'max_depth': 3,
    'learning_rate' : 0.1,
    'n_estimators' : 100,
    'random_state': 42
}

xgb_model = xgb.XGBClassifier(**params)
xgb_model.fit(x_scaled, y_train)
predxgb = xgb_model.predict(x_test_scaled)
xgb_accuracy_score = accuracy_score(y_test, predxgb)

print('Accuarcy of model is:', xgb_accuracy_score * 100)

#Accuracy: 89.03%

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay


# Predict positive class
pred_proba_xgb = xgb_model.predict_proba(x_test_scaled)[:, 1]

# ROC AUC score
roc_auc = roc_auc_score(y_test, pred_proba_xgb)
print(f'ROC AUC Score: {roc_auc:.4f}')
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_estimator(xgb_model, x_test_scaled, y_test, ax=ax, name='XGBoost')
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
ax.set_title('ROC Curve')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

#Classification Report
print("\nClassification Report:")
print(classification_report(y_test, predxgb))

#Confusion Matrix
cm = confusion_matrix(y_test, predxgb, labels=xgb_model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=xgb_model.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

# XG Boost Grid and Random Search

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import randint

#Grid Search

param_grid = {
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100,1000],
}
grid_search = GridSearchCV(xgb_model, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid_search.fit(x_scaled, y_train)
print("Grid Search Best Params:")
print(grid_search.best_params_)
test_predictions = grid_search.predict(x_test_scaled)
final_test_accuracy = accuracy_score(y_test, test_predictions)
print(f"Final Test Accuracy: {final_test_accuracy * 100:.2f}%")
print("ROC Score:", grid_search.best_score_)

In [ ]:
# Random Search

param_dist = {
    'max_depth': randint(1, 10),
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': randint(50, 1000),
    'subsample': [0.6, 0.8,0.9, 1.0],
    'colsample_bytree': [0.6, 0.8,0.9, 1.0],
    'gamma': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_lambda': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
}
random_search = RandomizedSearchCV(xgb_model, param_dist, n_iter=10, cv=5, scoring='roc_auc', n_jobs=-1)
random_search.fit(x_scaled, y_train)
print("Random Search Best Params:")
print(random_search.best_params_)
test_predictions = random_search.predict(x_test_scaled)
final_test_accuracy = accuracy_score(y_test, test_predictions)
print(f"Final Test Accuracy: {final_test_accuracy * 100:.2f}%")
print("ROC Score:", random_search.best_score_)

In [ ]:
#XG Boost Model with new parameters

new_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'random_state': 42,
    'colsample_bytree': 0.6,
    'learning_rate': 0.1,
    'max_depth': 1,
    'n_estimators': 592,
    'subsample': 0.8,
    'gamma': 0.5,
    'reg_alpha': 0.0,
    'reg_lambda': 0.1
}

new_xgb_model = xgb.XGBClassifier(**new_params)
new_xgb_model.fit(x_scaled, y_train)
newpredxgb = new_xgb_model.predict(x_test_scaled)
newxgb_accuracy_score = accuracy_score(y_test, newpredxgb)

print('Accuarcy of model is:', newxgb_accuracy_score * 100)
print("ROC Score:", reg_random_search.best_score_)

Accuracy with Random Search : 89.12%

ROC Score: 0.9503

In [ ]:
#Classification Report
new_pred_proba_xgb = xgb_model.predict_proba(x_test_scaled)[:, 1]
print("\nClassification Report:")
print(classification_report(y_test, new_pred_proba_xgb))

#Confusion Matrix
cm = confusion_matrix(y_test, new_pred_proba_xgb, labels=xgb_model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=new_pred_proba_xgb.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

#ADA Boosting

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada_model = AdaBoostClassifier(n_estimators=100, random_state=42)

y_pred_ada = ada_model.fit(x_scaled, y_train).predict(x_test_scaled)

accuracy_ada = accuracy_score(y_test, y_pred_ada)

print(f'AdaBoost Accuracy: {accuracy_ada * 100:.2f}%')

In [ ]:
grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
}
grid_search = GridSearchCV(ada_model, grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(x_scaled, y_train)
print("Grid Search Best Params:")
print(grid_search.best_params_)
test_predictions = grid_search.predict(x_test_scaled)
final_test_accuracy = accuracy_score(y_test, test_predictions)
print(f"Final Test Accuracy: {final_test_accuracy * 100:.2f}%")
print("ROC Score:", grid_search.best_score_)

In [ ]:
#Random Search


#Light GBM

In [ ]:
import lightgbm as lgb

lgb_model_gbt = lgb.LGBMClassifier(
    boosting_type='gbdt',
    n_estimators = 100,
    drop_rate = 0.1,
    skip_drop = 0.5,
    random_state=42)

y_pred_lgb_gbt = lgb_model_gbt.fit(x_scaled, y_train).predict(x_test_scaled)

accuracy_lgb_gbt = accuracy_score(y_test, y_pred_lgb_gbt)

print(f'LightGBM GBDT Accuracy: {accuracy_lgb_gbt * 100:.2f}%')

lgb_model_goss = lgb.LGBMClassifier(
    boosting_type='goss',
    n_estimators = 100,
    drop_rate = 0.1,
    skip_drop = 0.5,
    random_state=42)

y_pred_lgb_goss = lgb_model_goss.fit(x_scaled, y_train).predict(x_test_scaled)

accuracy_lgb_goss = accuracy_score(y_test, y_pred_lgb_goss)

print(f'Light GBM Goss Accuracy: {accuracy_lgb_gbt * 100:.2f}%')

lgb_model_dart = lgb.LGBMClassifier(
    boosting_type='dart',
    n_estimators = 100,
    drop_rate = 0.1,
    skip_drop = 0.5,
    random_state=42)

y_pred_lgb_dart = lgb_model_dart.fit(x_scaled, y_train).predict(x_test_scaled)

accuracy_lgb_dart = accuracy_score(y_test, y_pred_lgb_dart)

print(f'Light GBM Dart Accuracy: {accuracy_lgb_dart * 100:.2f}%')


##Voting Emsemble

In [ ]:
from sklearn.ensemble import VotingRegressor

model1 = new_xgb_model
model2 = ada_model
model3 = lgb_model_gbt
model4 = lrmodel

voting_regressor = VotingRegressor(estimators=[('XG_Boost', model1), ('ADA_Boost', model2), ('LightXGM_Boost', model3), ('LogReg', model4)])

voting_regressor.fit(x_scaled, y_train)

y_pred_voting = voting_regressor.predict(x_test_scaled)

voting_accuracy = accuracy_score(y_test, y_pred_voting)

print(f'Voting Ensemble Accuracy: {voting_accuracy * 100:.2f}%')

# Neural Network


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from torchvision.transforms import v2
import torch
import optuna

In [ ]:
print(f"PyTorch Version: {torch.__version__}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
X_train_tensor = torch.tensor(x_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
X_test_tensor = torch.tensor(x_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

L_train = TensorDataset(X_train_tensor, y_train_tensor)
L_test = TensorDataset(X_test_tensor, y_test_tensor)

train_dataloader = DataLoader(L_train, batch_size=64, shuffle=True)

linear_model = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1],16),
    nn.ReLU(),
    nn.Linear(16,8),
    nn.ReLU(),
    nn.Linear(8,1),
    nn.Sigmoid()
).to(device)

output = linear_model(X_train_tensor[0].to(device))

print(f"Output shape: {output.shape}")
print(f"Output values: {output}")

criteron = nn.BCELoss()
loss = criteron(output, y_train_tensor[0].unsqueeze(0).to(device)) # Unsqueeze to match output shape
print(f"Loss: {loss}")